In [21]:
import glob, json, joblib
import numpy as np, pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, classification_report)
from sklearn.utils.class_weight import compute_class_weight

files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print(files)

['/kaggle/input/datasets/swastisomwanshi/mine-subsidence-dataset/refined_subsidence_data (1).csv']


In [22]:
TARGET = "subsidence_rate_mm_day"

In [24]:
df = pd.read_csv(files[0], parse_dates=['timestamp']).sort_values('timestamp')
df = df.drop_duplicates().dropna(subset=[TARGET]).reset_index(drop=True)

DROP = ['timestamp',
        'danger_level',        # just a threshold on the target -> leakage
        'confidence_level',    # leaks the target, not a real sensor
        'cumulative_drift',    # running counter, acts as a clock
        'vertical_settle_est', 'horizontal_shift_est']  # exact functions of tilt

y = df[TARGET].values.astype(np.float32)
X = df.drop(columns=[TARGET] + DROP).astype(np.float32)
feature_cols = list(X.columns)
is_clf = False   # regression on subsidence rate (mm/day)

# Chronological split: train on the first 80%, test on the most recent 20%
split = int(len(df) * 0.8)
X_train, X_test = X.values[:split], X.values[split:]
y_train, y_test = y[:split], y[split:]
test_danger = df['danger_level'].values[split:]

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

y_log = np.log1p(y_train)
y_mean, y_std = float(y_log.mean()), float(y_log.std() + 1e-8)
y_train_s = (y_log - y_mean) / y_std
print(X_train.shape, X_test.shape, len(feature_cols), "features")

(1598, 29) (400, 29) 29 features


In [25]:
if is_clf:
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                class_weight='balanced', random_state=42).fit(X_train, y_train)
    print("RF weighted F1:", f1_score(y_test, rf.predict(X_test), average='weighted'))
else:
    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42).fit(X_train, y_train)
    p = rf.predict(X_test)
    print("RF MAE:", mean_absolute_error(y_test, p),
          "RMSE:", mean_squared_error(y_test, p) ** 0.5,
          "R2:", r2_score(y_test, p))

imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.head(10))   # fewer inputs = lighter edge model

RF MAE: 1.118823303779686 RMSE: 1.9116624079661544 R2: 0.8338018531868213
tilt_degrees_mean_6h         0.424196
tilt_degrees_mean_3h         0.310162
tilt_humidity_interaction    0.057405
temperature_celsius          0.029531
pressure_kpa_mean_6h         0.025185
pressure_kpa                 0.022655
tilt_magnitude_radians       0.011088
tilt_degrees                 0.010012
tilt_degrees_mean_24h        0.009908
tilt_magnitude_ewma          0.009390
dtype: float64


In [26]:
def build_model(n_features):
    inp = tf.keras.Input(shape=(n_features,))
    x = tf.keras.layers.Dense(64, activation='relu')(inp)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)
    if is_clf:
        out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
        m = tf.keras.Model(inp, out)
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    else:
        out = tf.keras.layers.Dense(1)(x)
        m = tf.keras.Model(inp, out)
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

model = build_model(X_train.shape[1])
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5),
]

fit_kwargs = dict(validation_split=0.2, epochs=200, batch_size=32,
                  callbacks=callbacks, verbose=2)
if is_clf:
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    history = model.fit(X_train, y_train, class_weight=dict(enumerate(cw)), **fit_kwargs)
else:
    history = model.fit(X_train, y_train_s, **fit_kwargs)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 29)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,545 (17.75 KB)

 Trainable params: 4,545 (17.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
40/40 - 2s - 44ms/step - loss: 0.4914 - mae: 0.5590 - val_loss: 0.2613 - val_mae: 0.3964 - learning_rate: 0.0010
Epoch 2/200
40/40 - 0s - 5ms/step - loss: 0.3098 - mae: 0.4312 - val_loss: 0.2466 - val_mae: 0.3906 - learning_rate: 0.0010
Epoch 3/200
40/40 - 0s - 5ms/step - loss: 0.2873 - mae: 0.4197 - val_loss: 0.2245 - val_mae: 0.3702 - learning_rate: 0.0010
Epoch 4/200
40/40 - 0s - 5ms/step - loss: 0.2593 - mae: 0.3940 - val_loss: 0.2150 - val_mae: 0.3581 - learning_rate: 0.0010
Epoch 5/200
40/40 - 0s - 5ms/step - loss: 0.2545 - mae: 0.3859 - val_loss: 0.2061 - val_mae: 0.3457 - learning_rate: 0.0010
Epoch 6/200
40/40 - 0s - 5ms/step - loss: 0.2457 - mae: 0.3779 - val_loss: 0.2056 - val_mae: 0.3427 - learning_rate: 0.0010
Epoch 7/200
40/40 - 0s - 5ms/step - loss: 0.2310 - mae: 0.3606 - val_loss: 0.1978 - val_mae: 0.3426 - learning_rate: 0.0010
Epoch 8/200
40/40 - 0s - 5ms/step - loss: 0.2217 - mae: 0.3581 - val_loss: 0.1972 - val_mae: 0.3314 - learning_rate: 0.0010
Epoch 9

In [28]:
if is_clf:
    pred = model.predict(X_test).argmax(axis=1)
    print("Accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred, target_names=[str(c) for c in le.classes_]))
else:
    pred = np.expm1(model.predict(X_test).ravel() * y_std + y_mean)
    print("MAE:", mean_absolute_error(y_test, pred),
          "RMSE:", mean_squared_error(y_test, pred) ** 0.5,
          "R2:", r2_score(y_test, pred))
def to_level(mm):
    return np.where(mm < 0.5, 'SAFE', np.where(mm < 5, 'WARNING', 'DANGER'))

print("Danger-level accuracy:", accuracy_score(test_danger, to_level(pred)))
print(classification_report(test_danger, to_level(pred)))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
MAE: 1.3598263263702393 RMSE: 2.550428848108713 R2: 0.7041781544685364
Danger-level accuracy: 0.72
              precision    recall  f1-score   support

      DANGER       0.83      0.94      0.88        81
        SAFE       0.89      0.69      0.78       229
     WARNING       0.42      0.60      0.49        90

    accuracy                           0.72       400
   macro avg       0.71      0.74      0.72       400
weighted avg       0.77      0.72      0.73       400



In [31]:
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
except Exception:
    model.export("saved_model_dir")
    converter = tf.lite.TFLiteConverter.from_saved_model("saved_model_dir")

converter.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantization
tflite_model = converter.convert()

with open("/kaggle/working/subsidence_model.tflite", "wb") as f:
    f.write(tflite_model)
print(f"TFLite size: {len(tflite_model)/1024:.1f} KB")

# Verify the TFLite model gives the same quality
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]

preds = []
for row in X_test:
    interp.set_tensor(inp_d['index'], row[None, :])
    interp.invoke()
    preds.append(interp.get_tensor(out_d['index'])[0])
preds = np.array(preds)

if is_clf:
    print("TFLite accuracy:", accuracy_score(y_test, preds.argmax(axis=1)))
else:
    p = np.expm1(preds.ravel() * y_std + y_mean)
    print("TFLite MAE:", mean_absolute_error(y_test, p), "R2:", r2_score(y_test, p))
def to_level(mm):
    return np.where(mm < 0.5, 'SAFE', np.where(mm < 5, 'WARNING', 'DANGER'))

p = np.maximum(p, 0)   # a rate can't be negative
print("TFLite danger-level accuracy:", accuracy_score(test_danger, to_level(p)))
print(classification_report(test_danger, to_level(p), zero_division=0))

INFO:tensorflow:Assets written to: /tmp/tmpks8oh8bd/assets


INFO:tensorflow:Assets written to: /tmp/tmpks8oh8bd/assets


Saved artifact at '/tmp/tmpks8oh8bd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 29), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136539430605968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430603472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430603088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430603280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430602704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430607120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430605008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136539430602320: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1789894753.466341      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789894753.466385      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


TFLite size: 9.7 KB
TFLite MAE: 1.3629043102264404 R2: 0.700731635093689
TFLite danger-level accuracy: 0.7225
              precision    recall  f1-score   support

      DANGER       0.83      0.94      0.88        81
        SAFE       0.89      0.69      0.78       229
     WARNING       0.42      0.60      0.49        90

    accuracy                           0.72       400
   macro avg       0.71      0.74      0.72       400
weighted avg       0.77      0.72      0.74       400



/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [38]:
meta = {
    "features": feature_cols,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "task": "classification" if is_clf else "regression",
}
if is_clf:
    meta["classes"] = [str(c) for c in le.classes_]
else:
    meta["y_mean"], meta["y_std"] = y_mean, y_std

json.dump(meta, open("/kaggle/working/model_meta.json", "w"), indent=2)